In [12]:
!pip install ssgetpy

In [16]:
import os
import tarfile
import numpy as np
import scipy.sparse as sp

from scipy.io import mmread
import ssgetpy

# ============================================================
# USER CONFIGURATION
# ============================================================

MATRIX_NAME = "scfxm1-2b"

OUTPUT_FILE = "test.h"

#NUM_CORES = 4

# Random reproducibility
np.random.seed(0)

# ------------------------------------------------------------
# Tile search space
# ------------------------------------------------------------
TILE_CANDIDATES = [
    4,
    8,
    16,
    32,
    64,
    128,
    256
]

# ------------------------------------------------------------
# L1 budget for double-buffer valcol
# (adjust according to MAGIA L1 capacity)
# ------------------------------------------------------------
MAX_DOUBLE_BUFFER_BYTES = 256 * 1024

# ============================================================
# PACKING FORMAT
# ============================================================

def pack_entry(value, col):

    value_u16 = np.uint16(np.int16(value))

    return (
        (np.uint32(col) << 16) |
        np.uint32(value_u16)
    )


# ============================================================
# HELPER
# ============================================================

def array_to_c_initializer(arr, per_line=16):

    flat = np.asarray(arr).flatten()

    lines = []

    for i in range(0, len(flat), per_line):

        chunk = flat[i:i + per_line]

        line = ", ".join(str(int(x)) for x in chunk)

        lines.append(line)

    return ",\n".join(lines)

# ============================================================
# DOWNLOAD MATRIX
# ============================================================

print(f"\nDownloading matrix: {MATRIX_NAME}")

result = ssgetpy.search(name=MATRIX_NAME)

if len(result) == 0:
    raise RuntimeError("Matrix not found.")

matrix = result[0]

os.makedirs("matrices", exist_ok=True)

matrix_dir = os.path.join("matrices", MATRIX_NAME)

os.makedirs(matrix_dir, exist_ok=True)

archive_path = os.path.join(
    "matrices",
    f"{MATRIX_NAME}.tar.gz"
)

# Remove existing archive if it's corrupted
if os.path.exists(archive_path):
    print(f"Removing existing corrupted archive: {archive_path}")
    os.remove(archive_path)

matrix.download(destpath="matrices")


if os.path.exists(archive_path):

    print(f"Extracting: {archive_path}")

    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(matrix_dir)

# ============================================================
# FIND MTX
# ============================================================

mtx_file = None

for root, dirs, files in os.walk(matrix_dir):

    for f in files:

        if (
            f.endswith(".mtx")
            and not f.endswith("_b.mtx")
            and not f.endswith("_x.mtx")
            ):

            mtx_file = os.path.join(root, f)

            break

    if mtx_file:
        break

if mtx_file is None:
    raise RuntimeError("Could not find .mtx file.")

print(f"Loading matrix from: {mtx_file}")

# ============================================================
# LOAD MATRIX
# ============================================================

A = mmread(mtx_file)

A = sp.csr_matrix(A)

# ============================================================
# MATRIX INFO
# ============================================================

M, N = A.shape

NNZ = A.nnz

total_elements = M * N

sparsity = 100.0 * (
    1.0 - (NNZ / total_elements)
)

print("\n================ MATRIX INFO ================")

print(f"Rows               : {M}")
print(f"Cols               : {N}")
print(f"NNZ                : {NNZ}")
print(f"Sparsity           : {sparsity:.2f}%")

# ============================================================
# QUANTIZE VALUES
# ============================================================

values = A.data.astype(np.float64)

scale = 16384.0 / np.max(np.abs(values))

values = np.round(values * scale).astype(np.int16)

# Avoid zero after quantization
#values[values == 0] = 1

A.data = values

# ============================================================
# AUTO TILE SIZE SEARCH
# ============================================================

print("\n================ TILE SEARCH ================")

best_tile_rows = None
best_max_tile_nnz = None
best_score = None

for tile_rows in TILE_CANDIDATES:

    max_tile_nnz = 0

    total_tile_nnz = 0

    num_tiles = 0

    valid = True

    for r in range(0, M, tile_rows):

        r_end = min(r + tile_rows, M)

        tile_nnz = (
            A.indptr[r_end] -
            A.indptr[r]
        )

        max_tile_nnz = max(
            max_tile_nnz,
            tile_nnz
        )

        total_tile_nnz += tile_nnz

        num_tiles += 1

    # --------------------------------------------------------
    # Double-buffer memory
    # --------------------------------------------------------
    double_buffer_bytes = (
        2 *
        max_tile_nnz *
        4
    )

    if double_buffer_bytes > MAX_DOUBLE_BUFFER_BYTES:
        valid = False

    avg_tile_nnz = total_tile_nnz / num_tiles

    imbalance = max_tile_nnz / avg_tile_nnz

    # --------------------------------------------------------
    # Simple heuristic:
    # lower imbalance is better
    # larger tiles are better
    # --------------------------------------------------------
    score = imbalance - (tile_rows * 0.0001)

    if valid:

        print(
            f"TILE_ROWS={tile_rows:<4} "
            f"MAX_TILE_NNZ={max_tile_nnz:<8} "
            f"DBUF={double_buffer_bytes/1024:.2f} KB "
            f"IMBALANCE={imbalance:.2f}"
        )

        if best_score is None or score < best_score:

            best_score = score

            best_tile_rows = tile_rows

            best_max_tile_nnz = max_tile_nnz

if best_tile_rows is None:
    raise RuntimeError("No valid tile size found.")

TILE_ROWS = best_tile_rows
MAX_TILE_NNZ = best_max_tile_nnz

print("\n================ BEST TILE ================")

print(f"TILE_ROWS         : {TILE_ROWS}")
print(f"MAX_TILE_NNZ      : {MAX_TILE_NNZ}")

# ============================================================
# GENERATE INPUT VECTOR
# ============================================================

x = np.random.randint(-4, 5, size=N, dtype=np.int16)

# ============================================================
# GOLDEN OUTPUT
# ============================================================

y_expected = A.dot(
    x.astype(np.int32)
).astype(np.int32)

# ============================================================
# BUILD PACKED CSR
# ============================================================

print("\nBuilding packed CSR...")

packed_valcol = np.zeros(
    NNZ,
    dtype=np.uint32
)

for i in range(NNZ):

    value = int(A.data[i])

    col = int(A.indices[i])

    packed_valcol[i] = pack_entry(
        value,
        col
    )

# ============================================================
# ROWPTR
# ============================================================

if M >= 131070:
    raise RuntimeError(
        "rowptr uint16 overflow. "
        "Use uint32_t rowptr."
    )

rowptr = A.indptr.astype(np.uint32)

# ============================================================
# MEMORY INFO
# ============================================================

dense_size_bytes = M * N * 2

compressed_size_bytes = (
    NNZ * 4 +
    (M + 1) * 4
)

double_buffer_bytes = (
    2 *
    MAX_TILE_NNZ *
    4
)

vector_bytes = (
    N * 2 +
    M * 4
)

occupied_memory_bytes = (
    compressed_size_bytes +
    double_buffer_bytes +
    vector_bytes
)

compression_ratio = (
    dense_size_bytes /
    compressed_size_bytes
)

print("\n================ MEMORY INFO ================")

print(f"Dense Matrix Size      : {dense_size_bytes / 1024:.2f} KB")

print(f"Compressed CSR Size    : {compressed_size_bytes / 1024:.2f} KB")

print(f"Double Buffer Size     : {double_buffer_bytes / 1024:.2f} KB")

print(f"Vector Storage         : {vector_bytes / 1024:.2f} KB")

print(f"Total Runtime Memory   : {occupied_memory_bytes / 1024:.2f} KB")

print(f"Compression Ratio      : {compression_ratio:.2f}x")

# ============================================================
# GENERATE test.h
# ============================================================

print("\nGenerating test.h ...")

with open(OUTPUT_FILE, "w") as f:

    f.write("#ifndef TEST_H\n")
    f.write("#define TEST_H\n\n")

    f.write("#include <stdint.h>\n\n")

    f.write("/*\n")
    f.write("==================================================\n")
    f.write("AUTO-GENERATED FILE\n")
    f.write("Packed Compact CSR Format\n")
    f.write("==================================================\n")
    f.write(f"Matrix Name            : {MATRIX_NAME}\n")
    f.write(f"Rows                   : {M}\n")
    f.write(f"Cols                   : {N}\n")
    f.write(f"NNZ                    : {NNZ}\n")
    f.write(f"Sparsity               : {sparsity:.2f} %\n")
    f.write(f"TILE_ROWS              : {TILE_ROWS}\n")
    f.write(f"MAX_TILE_NNZ           : {MAX_TILE_NNZ}\n")
    f.write(f"Dense Size             : {dense_size_bytes / 1024:.2f} KB\n")
    f.write(f"Compressed CSR Size    : {compressed_size_bytes / 1024:.2f} KB\n")
    f.write(f"Double Buffer Size     : {double_buffer_bytes / 1024:.2f} KB\n")
    f.write(f"Total Runtime Memory   : {occupied_memory_bytes / 1024:.2f} KB\n")
    f.write(f"Compression Ratio      : {compression_ratio:.2f}x\n")
    f.write("==================================================\n")
    f.write("*/\n\n")

    f.write(f"#define M {M}\n")
    f.write(f"#define N {N}\n")
    f.write(f"#define NNZ {NNZ}\n")
    #f.write(f"#define NUM_CORES {NUM_CORES}\n\n")

    f.write(f"#define TILE_ROWS {TILE_ROWS}\n")
    f.write(f"#define MAX_TILE_NNZ {MAX_TILE_NNZ}\n\n")

    # --------------------------------------------------------
    # Packed CSR
    # --------------------------------------------------------

    f.write(
        "static uint32_t valcol_l2[NNZ] "
        "__attribute__((section(\".l2\"), aligned(64))) = {\n"
    )

    f.write(
        array_to_c_initializer(
            packed_valcol,
            per_line=8
        )
    )

    f.write("\n};\n\n")

    # --------------------------------------------------------
    # rowptr
    # --------------------------------------------------------

    f.write(
        "static uint32_t rowptr_l2[M + 1] "
        "__attribute__((section(\".l2\"), aligned(64))) = {\n"
    )

    f.write(
        array_to_c_initializer(
            rowptr
        )
    )

    f.write("\n};\n\n")

    # --------------------------------------------------------
    # x vector
    # --------------------------------------------------------

    f.write(
        "static int16_t x[N] "
        "__attribute__((section(\".l2\"), aligned(64))) = {\n"
    )

    f.write(
        array_to_c_initializer(
            x
        )
    )

    f.write("\n};\n\n")

    # --------------------------------------------------------
    # output vector
    # --------------------------------------------------------

    f.write(
        "static int32_t y[M] "
        "__attribute__((section(\".l2\"), aligned(64))) = {0};\n\n"
    )

    # --------------------------------------------------------
    # golden output
    # --------------------------------------------------------

    f.write(
        "static int32_t y_expected[M] "
        "__attribute__((section(\".l2\"), aligned(64))) = {\n"
    )

    f.write(
        array_to_c_initializer(
            y_expected
        )
    )

    f.write("\n};\n\n")

    f.write("#endif\n")

print("\n============================================")
print(f"Generated: {OUTPUT_FILE}")
print("============================================")

scfxm1-2b:   0%|          | 0/915691 [00:00<?, ?B/s]

Extracting: matrices/scfxm1-2b.tar.gz
Loading matrix from: matrices/scfxm1-2b/scfxm1-2b/scfxm1-2b_b_3.mtx

================ MATRIX INFO ================
Rows               : 19036
Cols               : 1
NNZ                : 8860
Sparsity           : 53.46%

================ TILE SEARCH ================
TILE_ROWS=4    MAX_TILE_NNZ=4        DBUF=0.03 KB IMBALANCE=2.15
TILE_ROWS=8    MAX_TILE_NNZ=8        DBUF=0.06 KB IMBALANCE=2.15
TILE_ROWS=16   MAX_TILE_NNZ=16       DBUF=0.12 KB IMBALANCE=2.15
TILE_ROWS=32   MAX_TILE_NNZ=30       DBUF=0.23 KB IMBALANCE=2.01
TILE_ROWS=64   MAX_TILE_NNZ=54       DBUF=0.42 KB IMBALANCE=1.82
TILE_ROWS=128  MAX_TILE_NNZ=69       DBUF=0.54 KB IMBALANCE=1.16
TILE_ROWS=256  MAX_TILE_NNZ=138      DBUF=1.08 KB IMBALANCE=1.17

================ BEST TILE ================
TILE_ROWS         : 256
MAX_TILE_NNZ      : 138

Building packed CSR...

================ MEMORY INFO ================
Dense Matrix Size      : 37.18 KB
Compressed CSR Size    : 108.97 KB
Double B

/tmp/ipykernel_13136/2844573691.py:112: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(matrix_dir)


In [17]:
from google.colab import drive
import shutil

# Mount Google Drive
drive.mount('/content/drive')

# Define the source and destination paths
source_file = OUTPUT_FILE  # This variable is already defined as 'test.h'
destination_folder = '/content/drive/MyDrive/Colab_Notebooks'  # <--- IMPORTANT: Change 'Your_Project_Folder' to your actual project folder name in Google Drive

# Create the destination folder in Google Drive if it doesn't exist
os.makedirs(destination_folder, exist_ok=True)

# Copy the file to Google Drive
shutil.copy(source_file, destination_folder)

print(f"'{source_file}' has been copied to '{destination_folder}' in your Google Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
'test.h' has been copied to '/content/drive/MyDrive/Colab_Notebooks' in your Google Drive.
